# 07 — Modelagem Baseline para `target_banco_ganhou`

Este notebook continua o fluxo iniciado no notebook `06_feature_engineering_target_banco_ganhou.ipynb`.

## Objetivo

Treinar e avaliar modelos baseline para prever:

```text
target_banco_ganhou = 1 → banco ganhou / êxito
target_banco_ganhou = 0 → banco perdeu / condenação
```

A leitura de negócio principal deste notebook é a **probabilidade de perda do banco**:

```text
prob_banco_perdeu = P(target_banco_ganhou = 0)
```

Esse score deve apoiar:

1. Priorização de processos com maior risco de perda.
2. Ranking de processos para análise humana.
3. Avaliação de captura financeira no top-k.
4. Comparação entre baseline linear e modelos mais fortes.
5. Geração de insumos para relatório executivo do jurídico.

---

## Escopo deste notebook

Este notebook cobre:

1. Leitura das ABTs criadas no notebook 06.
2. Validação de colunas mínimas.
3. Remoção defensiva de possíveis variáveis com leakage.
4. Split temporal.
5. Pipeline baseline com `sklearn` + `LogisticRegression`.
6. Calibração opcional das probabilidades.
7. CatBoost opcional.
8. Métricas estatísticas.
9. Métricas financeiras.
10. Curvas de avaliação.
11. Ranking final de processos por probabilidade de perda.
12. Salvamento de relatórios CSV, scores e modelos.

---

## Premissa importante

Este modelo **não deve ser vendido como verdade jurídica**.

A comunicação recomendada é:

> Score probabilístico de risco de perda baseado no histórico de processos encerrados, criado para apoiar priorização, análise humana e gestão de carteira jurídica.

## 1. Imports e configuração inicial

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

import matplotlib.pyplot as plt

try:
    import joblib
except ImportError:
    joblib = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

## 2. Parâmetros do notebook

Ajuste os parâmetros abaixo conforme o ambiente.

Por padrão, o notebook tenta carregar as ABTs geradas pelo notebook 06 em `data/processed/`.

In [ ]:
@dataclass
class NotebookConfig:
    project_root: Path = Path.cwd()
    target_col: str = "target_banco_ganhou"
    date_col: str = "Datainicial"
    scenario_preferred: str = "scenario_a"  # opções: scenario_a, scenario_b, scenario_c, full
    use_text_feature: bool = True
    text_col: str = "fe_texto_curto_processo"
    test_size_quantile: float = 0.80
    calibration_size_quantile_within_train: float = 0.80
    threshold_loss: float = 0.50
    max_tfidf_features: int = 5000
    tfidf_min_df: int = 5
    top_k_percentages: Tuple[float, ...] = (0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00)
    enable_mlflow: bool = False
    save_models: bool = True

CFG = NotebookConfig()

DATA_DIR = CFG.project_root / "data" / "processed"
REPORT_DIR = CFG.project_root / "outputs" / "reports"
SCORE_DIR = CFG.project_root / "outputs" / "scores"
FIG_DIR = CFG.project_root / "outputs" / "figures"
MODEL_DIR = CFG.project_root / "models"

for path in [REPORT_DIR, SCORE_DIR, FIG_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

config_dict = asdict(CFG)
config_dict["project_root"] = str(config_dict["project_root"])
with open(REPORT_DIR / "07_modelagem_config.json", "w", encoding="utf-8") as f:
    json.dump(config_dict, f, indent=2, ensure_ascii=False)

config_dict

## 3. Leitura da ABT

Arquivos esperados do notebook 06:

```text
data/processed/abt_features_target_banco_ganhou_full.parquet
data/processed/abt_features_scenario_a_baseline_seguro.parquet
data/processed/abt_features_scenario_b_score_atual_com_cautela.parquet
data/processed/abt_features_scenario_c_catboost.parquet
```

A recomendação inicial é começar pelo **Cenário A — baseline seguro**.

In [ ]:
INPUT_FILES = {
    "scenario_a": DATA_DIR / "abt_features_scenario_a_baseline_seguro.parquet",
    "scenario_b": DATA_DIR / "abt_features_scenario_b_score_atual_com_cautela.parquet",
    "scenario_c": DATA_DIR / "abt_features_scenario_c_catboost.parquet",
    "full": DATA_DIR / "abt_features_target_banco_ganhou_full.parquet",
}

def read_abt(config: NotebookConfig) -> pd.DataFrame:
    preferred_path = INPUT_FILES.get(config.scenario_preferred)
    candidates = [preferred_path] + [p for k, p in INPUT_FILES.items() if p != preferred_path]

    for path in candidates:
        if path is not None and path.exists():
            print(f"Lendo ABT: {path}")
            if path.suffix.lower() == ".parquet":
                return pd.read_parquet(path)
            if path.suffix.lower() == ".csv":
                return pd.read_csv(path)
            raise ValueError(f"Formato de arquivo não suportado: {path}")

    msg = "Nenhuma ABT encontrada. Verifique se o notebook 06 foi executado e salvou os arquivos em data/processed/."
    raise FileNotFoundError(msg)

df = read_abt(CFG)
print(df.shape)
df.head()

## 4. Validações mínimas

Aqui validamos:

1. Se o target existe.
2. Se o target é binário.
3. Se a coluna temporal existe.
4. Se a data inicial está em formato datetime.
5. Se há duplicidades relevantes.

In [ ]:
def validate_minimum_columns(df: pd.DataFrame, config: NotebookConfig) -> None:
    missing = []
    for col in [config.target_col, config.date_col]:
        if col not in df.columns:
            missing.append(col)

    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

validate_minimum_columns(df, CFG)

df = df.copy()
df[CFG.date_col] = pd.to_datetime(df[CFG.date_col], errors="coerce")

target_counts = df[CFG.target_col].value_counts(dropna=False).rename_axis("target").reset_index(name="qtd")
target_counts["pct"] = target_counts["qtd"] / target_counts["qtd"].sum()
display(target_counts)

target_counts.to_csv(REPORT_DIR / "07_target_distribution_input.csv", index=False)

print("Período da base:")
print("min:", df[CFG.date_col].min())
print("max:", df[CFG.date_col].max())

if df[CFG.target_col].isna().any():
    print("Removendo linhas com target nulo.")
    df = df.dropna(subset=[CFG.target_col]).copy()

df[CFG.target_col] = df[CFG.target_col].astype(int)

valid_targets = set(df[CFG.target_col].unique())
if not valid_targets.issubset({0, 1}):
    raise ValueError(f"Target precisa ser binário com valores 0/1. Valores encontrados: {valid_targets}")

if df[CFG.date_col].isna().any():
    print("Removendo linhas com data inicial inválida.")
    df = df.dropna(subset=[CFG.date_col]).copy()

print(df.shape)

## 5. Remoção defensiva de leakage

Mesmo que o notebook 06 já tenha removido variáveis proibidas, este notebook aplica uma segunda barreira defensiva.

A regra é remover:

- target;
- desfecho;
- motivo de encerramento;
- situação;
- datas de encerramento/sentença/trânsito/condenação;
- resultados processuais;
- textos de sentença/tutela;
- probabilidades ou predições externas de resultado;
- valores de acordo/condenação/arbitramento;
- qualquer campo claramente pós-desfecho.

> Observação: a remoção abaixo é deliberadamente conservadora.

In [ ]:
LEAKAGE_EXACT_COLS = {
    "target_banco_ganhou",
    "target_banco_perdeu",
    "desfecho_categoria",
    "valor_perdido",
    "valor_ganho",
    "Situacao",
    "Motivo_Encerramento",
    "Dataencerramento",
}

LEAKAGE_PATTERNS = [
    r"^target_",
    r"^desfecho",
    r"^valor_perdido$",
    r"^valor_ganho$",
    r"resultado",
    r"sentenca",
    r"sentença",
    r"condenacao",
    r"condenação",
    r"acordo_prob",
    r"procedente_prob",
    r"procedente_parc",
    r"dano_moral_prob",
    r"provavel_",
    r"provável_",
    r"texto_sentenca",
    r"texto_sentença",
    r"texto_tutela",
    r"transito",
    r"trânsito",
    r"arquivado_data",
    r"data_arquivado",
    r"data_condenacao",
    r"data_condenação",
    r"valor_do_acordo",
    r"valor_arbitrado",
    r"data_do_acordao",
    r"acordao",
    r"acórdão",
]

ID_AND_META_CANDIDATES = [
    "Numerodistribuicao",
    "numero_processo",
    "Identificador",
    "id",
    "ID",
    "processo_id",
    "row_id",
    CFG.date_col,
]

def find_leakage_columns(columns: List[str], target_col: str) -> List[str]:
    leakage_cols = set()
    for col in columns:
        if col == target_col:
            leakage_cols.add(col)
            continue

        if col in LEAKAGE_EXACT_COLS:
            leakage_cols.add(col)
            continue

        col_norm = str(col).lower()
        for pattern in LEAKAGE_PATTERNS:
            if re.search(pattern, col_norm, flags=re.IGNORECASE):
                leakage_cols.add(col)
                break

    return sorted(leakage_cols)

leakage_cols_found = find_leakage_columns(df.columns.tolist(), CFG.target_col)

pd.DataFrame({"coluna_removida_por_leakage": leakage_cols_found}).to_csv(
    REPORT_DIR / "07_leakage_cols_removed_defensive.csv", index=False
)

print(f"Total de colunas com possível leakage encontradas: {len(leakage_cols_found)}")
leakage_cols_found[:80]

## 6. Separação de features, target e metadados

A modelagem usa `target_banco_ganhou`, mas todas as métricas de negócio serão calculadas a partir de:

```text
prob_banco_perdeu = 1 - prob_banco_ganhou
```

Quando possível, o notebook preserva identificadores apenas para o arquivo final de ranking, não como features.

In [ ]:
def build_feature_lists(
    df: pd.DataFrame,
    config: NotebookConfig,
    leakage_cols: List[str],
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, Dict[str, List[str]]]:
    y = df[config.target_col].astype(int).copy()

    meta_cols = [c for c in ID_AND_META_CANDIDATES if c in df.columns]
    meta_cols = list(dict.fromkeys(meta_cols))  # remove duplicatas preservando ordem
    meta = df[meta_cols].copy() if meta_cols else pd.DataFrame(index=df.index)

    drop_cols = set(leakage_cols) | set(meta_cols) | {config.target_col}
    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols].copy()

    # Remove colunas completamente nulas
    all_null_cols = X.columns[X.isna().all()].tolist()
    if all_null_cols:
        X = X.drop(columns=all_null_cols)

    # Remove colunas constantes
    constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
    if constant_cols:
        X = X.drop(columns=constant_cols)

    text_cols = []
    if config.use_text_feature and config.text_col in X.columns:
        text_cols = [config.text_col]

    numeric_cols = [
        c for c in X.select_dtypes(include=["number", "bool"]).columns.tolist()
        if c not in text_cols
    ]

    categorical_cols = [
        c for c in X.columns
        if c not in numeric_cols and c not in text_cols
    ]

    feature_groups = {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "text_cols": text_cols,
        "all_null_cols_removed": all_null_cols,
        "constant_cols_removed": constant_cols,
        "meta_cols": meta_cols,
    }

    return X, y, meta, feature_groups

X, y, meta, feature_groups = build_feature_lists(df, CFG, leakage_cols_found)

summary_rows = []
for group_name, cols in feature_groups.items():
    summary_rows.append({"grupo": group_name, "qtd_colunas": len(cols)})
feature_summary = pd.DataFrame(summary_rows)
display(feature_summary)

feature_summary.to_csv(REPORT_DIR / "07_feature_groups_summary.csv", index=False)

with open(REPORT_DIR / "07_feature_groups.json", "w", encoding="utf-8") as f:
    json.dump(feature_groups, f, indent=2, ensure_ascii=False)

print("Shape X:", X.shape)
print("Shape y:", y.shape)

## 7. Split temporal

O split recomendado é temporal, não aleatório.

Neste notebook:

```text
Treino/teste:
- treino: processos com Datainicial até o quantil 80% da data inicial
- teste: processos após esse ponto de corte

Calibração:
- dentro do treino, separa-se a parte final temporal para calibração
```

Isso evita uma validação artificialmente otimista.

In [ ]:
def temporal_split(
    df: pd.DataFrame,
    X: pd.DataFrame,
    y: pd.Series,
    meta: pd.DataFrame,
    date_col: str,
    test_size_quantile: float,
) -> Dict[str, pd.DataFrame]:
    sort_idx = df[date_col].sort_values().index
    df_sorted = df.loc[sort_idx].copy()
    X_sorted = X.loc[sort_idx].copy()
    y_sorted = y.loc[sort_idx].copy()
    meta_sorted = meta.loc[sort_idx].copy()

    split_date = df_sorted[date_col].quantile(test_size_quantile)

    train_mask = df_sorted[date_col] <= split_date
    test_mask = df_sorted[date_col] > split_date

    out = {
        "df_train": df_sorted.loc[train_mask].copy(),
        "df_test": df_sorted.loc[test_mask].copy(),
        "X_train": X_sorted.loc[train_mask].copy(),
        "X_test": X_sorted.loc[test_mask].copy(),
        "y_train": y_sorted.loc[train_mask].copy(),
        "y_test": y_sorted.loc[test_mask].copy(),
        "meta_train": meta_sorted.loc[train_mask].copy(),
        "meta_test": meta_sorted.loc[test_mask].copy(),
        "split_date": split_date,
    }
    return out

split = temporal_split(
    df=df,
    X=X,
    y=y,
    meta=meta,
    date_col=CFG.date_col,
    test_size_quantile=CFG.test_size_quantile,
)

for k in ["X_train", "X_test", "y_train", "y_test"]:
    print(k, split[k].shape)

split_report = pd.DataFrame([
    {
        "split_date": split["split_date"],
        "train_min_date": split["df_train"][CFG.date_col].min(),
        "train_max_date": split["df_train"][CFG.date_col].max(),
        "test_min_date": split["df_test"][CFG.date_col].min(),
        "test_max_date": split["df_test"][CFG.date_col].max(),
        "n_train": len(split["df_train"]),
        "n_test": len(split["df_test"]),
        "target_train_ganhou_rate": split["y_train"].mean(),
        "target_test_ganhou_rate": split["y_test"].mean(),
        "target_train_perda_rate": 1 - split["y_train"].mean(),
        "target_test_perda_rate": 1 - split["y_test"].mean(),
    }
])

display(split_report)
split_report.to_csv(REPORT_DIR / "07_temporal_split_report.csv", index=False)

## 8. Split temporal interno para calibração

A calibração é feita com uma janela temporal dentro do treino.

Fluxo:

```text
treino completo
├── train_fit: usado para treinar o modelo base
└── valid_cal: usado para calibrar probabilidades
```

In [ ]:
def temporal_train_calibration_split(
    df_train: pd.DataFrame,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    date_col: str,
    calibration_size_quantile_within_train: float,
) -> Dict[str, pd.DataFrame]:
    sort_idx = df_train[date_col].sort_values().index
    df_sorted = df_train.loc[sort_idx].copy()
    X_sorted = X_train.loc[sort_idx].copy()
    y_sorted = y_train.loc[sort_idx].copy()

    cal_split_date = df_sorted[date_col].quantile(calibration_size_quantile_within_train)

    fit_mask = df_sorted[date_col] <= cal_split_date
    cal_mask = df_sorted[date_col] > cal_split_date

    return {
        "df_fit": df_sorted.loc[fit_mask].copy(),
        "df_cal": df_sorted.loc[cal_mask].copy(),
        "X_fit": X_sorted.loc[fit_mask].copy(),
        "X_cal": X_sorted.loc[cal_mask].copy(),
        "y_fit": y_sorted.loc[fit_mask].copy(),
        "y_cal": y_sorted.loc[cal_mask].copy(),
        "cal_split_date": cal_split_date,
    }

cal_split = temporal_train_calibration_split(
    df_train=split["df_train"],
    X_train=split["X_train"],
    y_train=split["y_train"],
    date_col=CFG.date_col,
    calibration_size_quantile_within_train=CFG.calibration_size_quantile_within_train,
)

print("X_fit:", cal_split["X_fit"].shape)
print("X_cal:", cal_split["X_cal"].shape)
print("cal_split_date:", cal_split["cal_split_date"])

## 9. Pipeline baseline com `sklearn`

Baseline recomendado:

```text
numéricas   → SimpleImputer + StandardScaler
categóricas → SimpleImputer + OneHotEncoder
texto curto → TfidfVectorizer
modelo      → LogisticRegression(class_weight="balanced")
```

Por que esse baseline é útil?

1. É simples de explicar.
2. Ajuda a detectar vazamentos e problemas de dados.
3. Gera coeficientes interpretáveis.
4. Serve como referência antes de CatBoost/LightGBM/XGBoost.

In [ ]:
def build_onehot_encoder() -> OneHotEncoder:
    # Compatibilidade entre versões do sklearn:
    # sklearn >= 1.2 usa sparse_output; versões antigas usam sparse.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def build_baseline_logistic_pipeline(
    numeric_cols: List[str],
    categorical_cols: List[str],
    text_cols: List[str],
    config: NotebookConfig,
) -> Pipeline:
    transformers = []

    if numeric_cols:
        numeric_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler(with_mean=False)),
            ]
        )
        transformers.append(("num", numeric_pipeline, numeric_cols))

    if categorical_cols:
        categorical_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", build_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_pipeline, categorical_cols))

    if text_cols:
        # Usa apenas uma coluna de texto consolidado.
        # O FunctionTransformer garante entrada 1D de strings para o TfidfVectorizer.
        text_col = text_cols[0]

        def clean_text_input(x):
            if isinstance(x, pd.DataFrame):
                s = x.iloc[:, 0]
            elif isinstance(x, pd.Series):
                s = x
            else:
                s = pd.Series(np.asarray(x).ravel())
            return s.fillna("").astype(str)

        text_pipeline = Pipeline(
            steps=[
                ("clean_text", FunctionTransformer(clean_text_input, validate=False)),
                ("tfidf", TfidfVectorizer(
                    max_features=config.max_tfidf_features,
                    min_df=config.tfidf_min_df,
                    ngram_range=(1, 2),
                    strip_accents="unicode",
                    lowercase=True,
                )),
            ]
        )
        transformers.append(("text", text_pipeline, text_col))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )

    clf = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="saga",
        max_iter=800,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", clf),
        ]
    )

    return pipeline

baseline_pipe = build_baseline_logistic_pipeline(
    numeric_cols=feature_groups["numeric_cols"],
    categorical_cols=feature_groups["categorical_cols"],
    text_cols=feature_groups["text_cols"],
    config=CFG,
)

baseline_pipe

## 10. Funções de avaliação

A modelagem treina o classificador com target:

```text
1 = banco ganhou
0 = banco perdeu
```

Mas a priorização de negócio usa:

```text
prob_banco_perdeu = P(target_banco_ganhou = 0)
```

In [ ]:
def get_binary_probabilities(model, X: pd.DataFrame) -> pd.DataFrame:
    proba = model.predict_proba(X)

    if not hasattr(model, "classes_"):
        raise AttributeError("Modelo não possui atributo classes_. Verifique se foi treinado.")

    classes = list(model.classes_)

    if 0 not in classes or 1 not in classes:
        raise ValueError(f"Classes esperadas 0/1, mas foram encontradas: {classes}")

    idx_0 = classes.index(0)
    idx_1 = classes.index(1)

    return pd.DataFrame({
        "prob_banco_perdeu": proba[:, idx_0],
        "prob_banco_ganhou": proba[:, idx_1],
    }, index=X.index)


def evaluate_binary_predictions(
    y_true_ganhou: pd.Series,
    prob_df: pd.DataFrame,
    threshold_loss: float = 0.50,
    prefix: str = "model",
) -> Dict[str, float]:
    y_true_ganhou = pd.Series(y_true_ganhou).astype(int)
    y_true_perda = 1 - y_true_ganhou

    p_ganhou = prob_df["prob_banco_ganhou"].values
    p_perda = prob_df["prob_banco_perdeu"].values

    pred_perda = (p_perda >= threshold_loss).astype(int)

    metrics = {
        "model": prefix,
        "n": len(y_true_ganhou),
        "ganhou_rate": float(y_true_ganhou.mean()),
        "perda_rate": float(y_true_perda.mean()),
        "roc_auc_ganhou": float(roc_auc_score(y_true_ganhou, p_ganhou)),
        "roc_auc_perda": float(roc_auc_score(y_true_perda, p_perda)),
        "pr_auc_perda": float(average_precision_score(y_true_perda, p_perda)),
        "brier_ganhou": float(brier_score_loss(y_true_ganhou, p_ganhou)),
        "log_loss": float(log_loss(y_true_ganhou, np.column_stack([p_perda, p_ganhou]), labels=[0, 1])),
        "threshold_loss": float(threshold_loss),
        "precision_perda_at_threshold": float(precision_score(y_true_perda, pred_perda, zero_division=0)),
        "recall_perda_at_threshold": float(recall_score(y_true_perda, pred_perda, zero_division=0)),
        "f1_perda_at_threshold": float(f1_score(y_true_perda, pred_perda, zero_division=0)),
    }

    return metrics


def confusion_matrix_loss_positive(
    y_true_ganhou: pd.Series,
    prob_df: pd.DataFrame,
    threshold_loss: float = 0.50,
) -> pd.DataFrame:
    y_true_perda = 1 - pd.Series(y_true_ganhou).astype(int)
    pred_perda = (prob_df["prob_banco_perdeu"].values >= threshold_loss).astype(int)

    cm = confusion_matrix(y_true_perda, pred_perda, labels=[0, 1])

    return pd.DataFrame(
        cm,
        index=["real_nao_perda", "real_perda"],
        columns=["pred_nao_perda", "pred_perda"],
    )


def resolve_exposure_col(df: pd.DataFrame) -> Optional[str]:
    candidates = [
        "Valorajuizado",
        "fe_valor_ajuizado",
        "valor_valor",
        "fe_valor_ajuizado_winsor",
        "fe_valor_ajuizado_log1p",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def financial_gain_table(
    df_eval: pd.DataFrame,
    y_true_ganhou: pd.Series,
    prob_df: pd.DataFrame,
    top_percentages: Tuple[float, ...],
    exposure_col: Optional[str] = None,
) -> pd.DataFrame:
    base = df_eval.copy()
    base["target_banco_ganhou"] = pd.Series(y_true_ganhou, index=df_eval.index).astype(int)
    base["target_banco_perdeu"] = 1 - base["target_banco_ganhou"]
    base["prob_banco_perdeu"] = prob_df["prob_banco_perdeu"].values
    base["prob_banco_ganhou"] = prob_df["prob_banco_ganhou"].values

    if exposure_col is None or exposure_col not in base.columns:
        base["exposure_proxy"] = 1.0
        exposure_col_used = "exposure_proxy"
    else:
        base["exposure_proxy"] = pd.to_numeric(base[exposure_col], errors="coerce").fillna(0).clip(lower=0)
        exposure_col_used = exposure_col

    base["loss_exposure_proxy"] = base["target_banco_perdeu"] * base["exposure_proxy"]

    base = base.sort_values("prob_banco_perdeu", ascending=False).reset_index(drop=True)

    total_n = len(base)
    total_losses = base["target_banco_perdeu"].sum()
    total_loss_exposure = base["loss_exposure_proxy"].sum()

    rows = []
    for pct in top_percentages:
        n_top = int(np.ceil(total_n * pct))
        n_top = max(1, min(n_top, total_n))

        top = base.iloc[:n_top].copy()
        rows.append({
            "top_pct": pct,
            "n_top": n_top,
            "losses_in_top": int(top["target_banco_perdeu"].sum()),
            "loss_rate_in_top": float(top["target_banco_perdeu"].mean()),
            "recall_loss_count": float(top["target_banco_perdeu"].sum() / total_losses) if total_losses else np.nan,
            "captured_loss_exposure_proxy": float(top["loss_exposure_proxy"].sum()),
            "captured_loss_exposure_proxy_pct": float(top["loss_exposure_proxy"].sum() / total_loss_exposure) if total_loss_exposure else np.nan,
            "mean_prob_banco_perdeu_in_top": float(top["prob_banco_perdeu"].mean()),
            "min_prob_banco_perdeu_in_top": float(top["prob_banco_perdeu"].min()),
            "exposure_col_used": exposure_col_used,
        })

    return pd.DataFrame(rows)


def decile_table_loss(
    df_eval: pd.DataFrame,
    y_true_ganhou: pd.Series,
    prob_df: pd.DataFrame,
    exposure_col: Optional[str] = None,
) -> pd.DataFrame:
    base = df_eval.copy()
    base["target_banco_ganhou"] = pd.Series(y_true_ganhou, index=df_eval.index).astype(int)
    base["target_banco_perdeu"] = 1 - base["target_banco_ganhou"]
    base["prob_banco_perdeu"] = prob_df["prob_banco_perdeu"].values

    if exposure_col is None or exposure_col not in base.columns:
        base["exposure_proxy"] = 1.0
        exposure_col_used = "exposure_proxy"
    else:
        base["exposure_proxy"] = pd.to_numeric(base[exposure_col], errors="coerce").fillna(0).clip(lower=0)
        exposure_col_used = exposure_col

    base["loss_exposure_proxy"] = base["target_banco_perdeu"] * base["exposure_proxy"]

    # Decil 1 = maior risco
    base = base.sort_values("prob_banco_perdeu", ascending=False).reset_index(drop=True)
    base["rank"] = np.arange(1, len(base) + 1)
    base["decil_risco"] = pd.qcut(base["rank"], q=10, labels=list(range(1, 11)))

    dec = (
        base.groupby("decil_risco", observed=True)
        .agg(
            n=("target_banco_perdeu", "size"),
            perda_rate=("target_banco_perdeu", "mean"),
            qtd_perdas=("target_banco_perdeu", "sum"),
            prob_banco_perdeu_min=("prob_banco_perdeu", "min"),
            prob_banco_perdeu_mean=("prob_banco_perdeu", "mean"),
            prob_banco_perdeu_max=("prob_banco_perdeu", "max"),
            exposure_proxy_sum=("exposure_proxy", "sum"),
            loss_exposure_proxy_sum=("loss_exposure_proxy", "sum"),
        )
        .reset_index()
    )
    dec["exposure_col_used"] = exposure_col_used
    return dec


def threshold_report_loss(
    y_true_ganhou: pd.Series,
    prob_df: pd.DataFrame,
    thresholds: Optional[List[float]] = None,
) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19).round(2).tolist()

    y_true_perda = 1 - pd.Series(y_true_ganhou).astype(int)
    p_perda = prob_df["prob_banco_perdeu"].values

    rows = []
    for thr in thresholds:
        pred_perda = (p_perda >= thr).astype(int)
        rows.append({
            "threshold_loss": thr,
            "precision_perda": precision_score(y_true_perda, pred_perda, zero_division=0),
            "recall_perda": recall_score(y_true_perda, pred_perda, zero_division=0),
            "f1_perda": f1_score(y_true_perda, pred_perda, zero_division=0),
            "qtd_pred_perda": int(pred_perda.sum()),
            "pct_pred_perda": float(pred_perda.mean()),
        })
    return pd.DataFrame(rows)

## 11. Treinamento do baseline

O primeiro modelo treinado é uma regressão logística regularizada.

A avaliação será feita no conjunto de teste temporal.

In [ ]:
print("Treinando baseline LogisticRegression...")

baseline_pipe.fit(split["X_train"], split["y_train"])

prob_test_baseline = get_binary_probabilities(baseline_pipe, split["X_test"])

metrics_baseline = evaluate_binary_predictions(
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_baseline,
    threshold_loss=CFG.threshold_loss,
    prefix="logistic_regression_baseline",
)

metrics_baseline_df = pd.DataFrame([metrics_baseline])
display(metrics_baseline_df.T)

metrics_baseline_df.to_csv(REPORT_DIR / "07_logistic_baseline_metrics.csv", index=False)

cm_baseline = confusion_matrix_loss_positive(
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_baseline,
    threshold_loss=CFG.threshold_loss,
)
display(cm_baseline)
cm_baseline.to_csv(REPORT_DIR / "07_logistic_confusion_matrix_loss_positive.csv")

## 12. Avaliação financeira do baseline

A pergunta de negócio é:

> Se o jurídico atuar nos top 1%, 5%, 10% e 20% processos com maior probabilidade de perda, quanto do risco histórico seria capturado?

A métrica financeira usa como proxy preferencial:

```text
Valorajuizado
```

Se `Valorajuizado` não existir, o notebook tenta outras colunas. Se nenhuma existir, usa contagem simples.

In [ ]:
exposure_col = resolve_exposure_col(split["df_test"])
print("Coluna usada como proxy de exposição financeira:", exposure_col)

gain_baseline = financial_gain_table(
    df_eval=split["df_test"],
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_baseline,
    top_percentages=CFG.top_k_percentages,
    exposure_col=exposure_col,
)

display(gain_baseline)
gain_baseline.to_csv(REPORT_DIR / "07_logistic_financial_gain_table.csv", index=False)

decile_baseline = decile_table_loss(
    df_eval=split["df_test"],
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_baseline,
    exposure_col=exposure_col,
)

display(decile_baseline)
decile_baseline.to_csv(REPORT_DIR / "07_logistic_decile_table_loss.csv", index=False)

threshold_baseline = threshold_report_loss(
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_baseline,
)
display(threshold_baseline)
threshold_baseline.to_csv(REPORT_DIR / "07_logistic_threshold_report_loss.csv", index=False)

## 13. Curvas do baseline

Curvas salvas:

```text
outputs/figures/07_logistic_roc_loss.png
outputs/figures/07_logistic_pr_loss.png
outputs/figures/07_logistic_calibration.png
outputs/figures/07_logistic_financial_gain.png
```

In [ ]:
def plot_roc_loss(y_true_ganhou, prob_df, output_path: Path, title: str):
    y_true_perda = 1 - pd.Series(y_true_ganhou).astype(int)
    fpr, tpr, _ = roc_curve(y_true_perda, prob_df["prob_banco_perdeu"])
    auc = roc_auc_score(y_true_perda, prob_df["prob_banco_perdeu"])

    plt.figure(figsize=(8, 5))
    plt.plot(fpr, tpr, label=f"ROC AUC perda = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Aleatório")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()


def plot_pr_loss(y_true_ganhou, prob_df, output_path: Path, title: str):
    y_true_perda = 1 - pd.Series(y_true_ganhou).astype(int)
    precision, recall, _ = precision_recall_curve(y_true_perda, prob_df["prob_banco_perdeu"])
    ap = average_precision_score(y_true_perda, prob_df["prob_banco_perdeu"])

    plt.figure(figsize=(8, 5))
    plt.plot(recall, precision, label=f"PR AUC perda = {ap:.4f}")
    plt.xlabel("Recall perda")
    plt.ylabel("Precision perda")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()


def plot_calibration(y_true_ganhou, prob_df, output_path: Path, title: str):
    frac_pos, mean_pred = calibration_curve(
        pd.Series(y_true_ganhou).astype(int),
        prob_df["prob_banco_ganhou"],
        n_bins=10,
        strategy="quantile",
    )

    plt.figure(figsize=(8, 5))
    plt.plot(mean_pred, frac_pos, marker="o", label="Modelo")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfeitamente calibrado")
    plt.xlabel("Probabilidade média prevista de ganho")
    plt.ylabel("Fração observada de ganho")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()


def plot_financial_gain(gain_df: pd.DataFrame, output_path: Path, title: str):
    plt.figure(figsize=(8, 5))
    plt.plot(
        gain_df["top_pct"] * 100,
        gain_df["captured_loss_exposure_proxy_pct"] * 100,
        marker="o",
    )
    plt.xlabel("Top % processos por probabilidade de perda")
    plt.ylabel("% exposição de perdas capturada")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()


plot_roc_loss(
    split["y_test"],
    prob_test_baseline,
    FIG_DIR / "07_logistic_roc_loss.png",
    "ROC — classe perda — Logistic Regression",
)

plot_pr_loss(
    split["y_test"],
    prob_test_baseline,
    FIG_DIR / "07_logistic_pr_loss.png",
    "Precision-Recall — classe perda — Logistic Regression",
)

plot_calibration(
    split["y_test"],
    prob_test_baseline,
    FIG_DIR / "07_logistic_calibration.png",
    "Calibração — probabilidade de ganho — Logistic Regression",
)

plot_financial_gain(
    gain_baseline,
    FIG_DIR / "07_logistic_financial_gain.png",
    "Ganho financeiro acumulado — Logistic Regression",
)

## 14. Calibração das probabilidades

A calibração melhora a leitura probabilística.

Exemplo:

```text
Entre os processos com score de perda próximo de 70%, aproximadamente 70% deveriam de fato ser perdas.
```

Aqui usamos:

- `train_fit`: treina o pipeline;
- `valid_cal`: calibra probabilidades;
- `test`: avalia fora do tempo.

Método padrão:

```text
isotonic
```

Se houver poucos dados, considere trocar para:

```text
sigmoid
```

In [ ]:
def build_calibrated_prefit_model(fitted_model, X_cal, y_cal, method: str = "isotonic"):
    try:
        calibrated = CalibratedClassifierCV(estimator=fitted_model, method=method, cv="prefit")
    except TypeError:
        calibrated = CalibratedClassifierCV(base_estimator=fitted_model, method=method, cv="prefit")

    calibrated.fit(X_cal, y_cal)
    return calibrated


print("Treinando pipeline base no train_fit para posterior calibração...")

baseline_for_calibration = clone(baseline_pipe)
baseline_for_calibration.fit(cal_split["X_fit"], cal_split["y_fit"])

calibrated_logistic = build_calibrated_prefit_model(
    fitted_model=baseline_for_calibration,
    X_cal=cal_split["X_cal"],
    y_cal=cal_split["y_cal"],
    method="isotonic",
)

prob_test_calibrated = get_binary_probabilities(calibrated_logistic, split["X_test"])

metrics_calibrated = evaluate_binary_predictions(
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_calibrated,
    threshold_loss=CFG.threshold_loss,
    prefix="logistic_regression_calibrated_isotonic",
)

metrics_calibrated_df = pd.DataFrame([metrics_calibrated])
display(metrics_calibrated_df.T)

metrics_calibrated_df.to_csv(REPORT_DIR / "07_logistic_calibrated_metrics.csv", index=False)

gain_calibrated = financial_gain_table(
    df_eval=split["df_test"],
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_calibrated,
    top_percentages=CFG.top_k_percentages,
    exposure_col=exposure_col,
)
display(gain_calibrated)
gain_calibrated.to_csv(REPORT_DIR / "07_logistic_calibrated_financial_gain_table.csv", index=False)

## 15. Comparação baseline vs calibrado

A calibração deve ser avaliada principalmente por:

```text
Brier Score
Log Loss
Curva de calibração
```

Ela pode melhorar a qualidade probabilística sem necessariamente aumentar ROC AUC.

In [ ]:
metrics_compare = pd.concat(
    [
        metrics_baseline_df,
        metrics_calibrated_df,
    ],
    ignore_index=True,
)

display(metrics_compare)

metrics_compare.to_csv(REPORT_DIR / "07_modeling_metrics_summary.csv", index=False)

plot_calibration(
    split["y_test"],
    prob_test_calibrated,
    FIG_DIR / "07_logistic_calibrated_calibration.png",
    "Calibração — probabilidade de ganho — Logistic Regression calibrada",
)

plot_financial_gain(
    gain_calibrated,
    FIG_DIR / "07_logistic_calibrated_financial_gain.png",
    "Ganho financeiro acumulado — Logistic Regression calibrada",
)

## 16. Ranking de processos por probabilidade de perda

Este é o principal artefato para consumo pelo time jurídico.

Campos esperados:

```text
Identificadores do processo
Datainicial
target_banco_ganhou
target_banco_perdeu
prob_banco_perdeu
prob_banco_ganhou
rank_risco_perda
decil_risco
proxy financeiro, quando disponível
```

In [ ]:
def build_ranking_output(
    df_eval: pd.DataFrame,
    meta_eval: pd.DataFrame,
    y_true_ganhou: pd.Series,
    prob_df: pd.DataFrame,
    exposure_col: Optional[str],
    model_name: str,
) -> pd.DataFrame:
    out = meta_eval.copy()

    # Caso meta esteja vazio, preserva índice para rastreabilidade
    if out.empty:
        out = pd.DataFrame(index=df_eval.index)
        out["index_original"] = df_eval.index

    out["target_banco_ganhou"] = pd.Series(y_true_ganhou, index=df_eval.index).astype(int).values
    out["target_banco_perdeu"] = 1 - out["target_banco_ganhou"]
    out["prob_banco_perdeu"] = prob_df["prob_banco_perdeu"].values
    out["prob_banco_ganhou"] = prob_df["prob_banco_ganhou"].values
    out["model_name"] = model_name

    if exposure_col is not None and exposure_col in df_eval.columns:
        out[exposure_col] = df_eval[exposure_col].values

    out = out.sort_values("prob_banco_perdeu", ascending=False).reset_index(drop=True)
    out["rank_risco_perda"] = np.arange(1, len(out) + 1)
    out["decil_risco"] = pd.qcut(out["rank_risco_perda"], q=10, labels=list(range(1, 11)))

    return out

ranking_baseline = build_ranking_output(
    df_eval=split["df_test"],
    meta_eval=split["meta_test"],
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_baseline,
    exposure_col=exposure_col,
    model_name="logistic_regression_baseline",
)

display(ranking_baseline.head(30))

ranking_baseline.to_csv(SCORE_DIR / "07_logistic_ranked_processes_test.csv", index=False)
try:
    ranking_baseline.to_parquet(SCORE_DIR / "07_logistic_ranked_processes_test.parquet", index=False)
except Exception as e:
    print("Não foi possível salvar parquet do ranking baseline:", e)

ranking_calibrated = build_ranking_output(
    df_eval=split["df_test"],
    meta_eval=split["meta_test"],
    y_true_ganhou=split["y_test"],
    prob_df=prob_test_calibrated,
    exposure_col=exposure_col,
    model_name="logistic_regression_calibrated_isotonic",
)

ranking_calibrated.to_csv(SCORE_DIR / "07_logistic_calibrated_ranked_processes_test.csv", index=False)
try:
    ranking_calibrated.to_parquet(SCORE_DIR / "07_logistic_calibrated_ranked_processes_test.parquet", index=False)
except Exception as e:
    print("Não foi possível salvar parquet do ranking calibrado:", e)

## 17. Interpretabilidade básica do baseline linear

Esta seção tenta recuperar os principais coeficientes da regressão logística.

Interpretação:

```text
coeficiente positivo  → aumenta probabilidade de target_banco_ganhou = 1
coeficiente negativo  → aumenta probabilidade de target_banco_ganhou = 0, ou seja, perda
```

Portanto, para risco de perda, olhe principalmente os coeficientes mais negativos.

In [ ]:
def get_feature_names_from_column_transformer(preprocessor: ColumnTransformer) -> List[str]:
    try:
        return preprocessor.get_feature_names_out().tolist()
    except Exception:
        feature_names = []
        for name, transformer, cols in preprocessor.transformers_:
            if name == "remainder" and transformer == "drop":
                continue

            if hasattr(transformer, "get_feature_names_out"):
                try:
                    if isinstance(cols, str):
                        names = transformer.get_feature_names_out([cols]).tolist()
                    else:
                        names = transformer.get_feature_names_out(cols).tolist()
                    feature_names.extend([f"{name}__{n}" for n in names])
                    continue
                except Exception:
                    pass

            if isinstance(cols, str):
                feature_names.append(f"{name}__{cols}")
            else:
                feature_names.extend([f"{name}__{c}" for c in cols])
        return feature_names


def logistic_coefficients_report(model_pipeline: Pipeline, top_n: int = 50) -> pd.DataFrame:
    preprocessor = model_pipeline.named_steps["preprocessor"]
    model = model_pipeline.named_steps["model"]

    feature_names = get_feature_names_from_column_transformer(preprocessor)
    coefs = model.coef_.ravel()

    n = min(len(feature_names), len(coefs))
    report = pd.DataFrame({
        "feature": feature_names[:n],
        "coef_target_ganhou": coefs[:n],
    })
    report["coef_risco_perda"] = -report["coef_target_ganhou"]

    report = report.sort_values("coef_risco_perda", ascending=False).reset_index(drop=True)
    return report

try:
    coef_report = logistic_coefficients_report(baseline_pipe, top_n=100)
    display(coef_report.head(50))
    coef_report.to_csv(REPORT_DIR / "07_logistic_coefficients.csv", index=False)
except Exception as e:
    print("Não foi possível gerar relatório de coeficientes:", e)

## 18. CatBoost opcional

O CatBoost costuma ser uma boa alternativa para bases jurídicas porque lida bem com:

- alta cardinalidade;
- categorias sujas;
- relações não lineares;
- interações implícitas.

Esta seção é opcional. Se `catboost` não estiver instalado, será ignorada.

> Atenção: CatBoost também deve respeitar a lista defensiva de leakage e o split temporal.

In [ ]:
try:
    from catboost import CatBoostClassifier, Pool

    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

print("CatBoost disponível?", CATBOOST_AVAILABLE)

In [ ]:
if CATBOOST_AVAILABLE:
    def prepare_catboost_data(
        X_train: pd.DataFrame,
        X_test: pd.DataFrame,
        feature_groups: Dict[str, List[str]],
    ):
        X_train_cb = X_train.copy()
        X_test_cb = X_test.copy()

        cat_cols = [
            c for c in X_train_cb.columns
            if c in feature_groups["categorical_cols"] or c in feature_groups["text_cols"]
        ]

        # CatBoost aceita string/categoria para cat_features.
        for c in cat_cols:
            X_train_cb[c] = X_train_cb[c].astype("string").fillna("__MISSING__")
            X_test_cb[c] = X_test_cb[c].astype("string").fillna("__MISSING__")

        # Numéricas: garantir coerção.
        for c in feature_groups["numeric_cols"]:
            if c in X_train_cb.columns:
                X_train_cb[c] = pd.to_numeric(X_train_cb[c], errors="coerce")
                X_test_cb[c] = pd.to_numeric(X_test_cb[c], errors="coerce")

        cat_feature_indices = [X_train_cb.columns.get_loc(c) for c in cat_cols if c in X_train_cb.columns]

        return X_train_cb, X_test_cb, cat_feature_indices

    X_train_cb, X_test_cb, cat_feature_indices = prepare_catboost_data(
        split["X_train"],
        split["X_test"],
        feature_groups,
    )

    cat_model = CatBoostClassifier(
        iterations=800,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=RANDOM_STATE,
        verbose=100,
        allow_writing_files=False,
    )

    cat_model.fit(
        X_train_cb,
        split["y_train"],
        cat_features=cat_feature_indices,
        eval_set=(X_test_cb, split["y_test"]),
        use_best_model=True,
    )

    prob_test_cat = get_binary_probabilities(cat_model, X_test_cb)

    metrics_cat = evaluate_binary_predictions(
        y_true_ganhou=split["y_test"],
        prob_df=prob_test_cat,
        threshold_loss=CFG.threshold_loss,
        prefix="catboost_baseline",
    )

    metrics_cat_df = pd.DataFrame([metrics_cat])
    display(metrics_cat_df.T)
    metrics_cat_df.to_csv(REPORT_DIR / "07_catboost_metrics.csv", index=False)

    gain_cat = financial_gain_table(
        df_eval=split["df_test"],
        y_true_ganhou=split["y_test"],
        prob_df=prob_test_cat,
        top_percentages=CFG.top_k_percentages,
        exposure_col=exposure_col,
    )
    display(gain_cat)
    gain_cat.to_csv(REPORT_DIR / "07_catboost_financial_gain_table.csv", index=False)

    ranking_cat = build_ranking_output(
        df_eval=split["df_test"],
        meta_eval=split["meta_test"],
        y_true_ganhou=split["y_test"],
        prob_df=prob_test_cat,
        exposure_col=exposure_col,
        model_name="catboost_baseline",
    )
    ranking_cat.to_csv(SCORE_DIR / "07_catboost_ranked_processes_test.csv", index=False)

    try:
        ranking_cat.to_parquet(SCORE_DIR / "07_catboost_ranked_processes_test.parquet", index=False)
    except Exception as e:
        print("Não foi possível salvar parquet do ranking CatBoost:", e)

    feature_importance_cat = pd.DataFrame({
        "feature": X_train_cb.columns,
        "importance": cat_model.get_feature_importance(),
    }).sort_values("importance", ascending=False)

    display(feature_importance_cat.head(50))
    feature_importance_cat.to_csv(REPORT_DIR / "07_catboost_feature_importance.csv", index=False)

    plot_roc_loss(
        split["y_test"],
        prob_test_cat,
        FIG_DIR / "07_catboost_roc_loss.png",
        "ROC — classe perda — CatBoost",
    )

    plot_pr_loss(
        split["y_test"],
        prob_test_cat,
        FIG_DIR / "07_catboost_pr_loss.png",
        "Precision-Recall — classe perda — CatBoost",
    )

    plot_financial_gain(
        gain_cat,
        FIG_DIR / "07_catboost_financial_gain.png",
        "Ganho financeiro acumulado — CatBoost",
    )

else:
    print("CatBoost não instalado. Se quiser rodar esta seção, instale: pip install catboost")

## 19. Consolidação comparativa dos modelos

Se CatBoost foi executado, ele será incluído na tabela consolidada.

In [ ]:
all_metrics = [metrics_baseline_df, metrics_calibrated_df]

if "metrics_cat_df" in globals():
    all_metrics.append(metrics_cat_df)

metrics_summary = pd.concat(all_metrics, ignore_index=True)
metrics_summary = metrics_summary.sort_values("pr_auc_perda", ascending=False).reset_index(drop=True)

display(metrics_summary)

metrics_summary.to_csv(REPORT_DIR / "07_all_models_metrics_summary.csv", index=False)

## 20. Salvamento dos modelos

Modelos salvos, quando `CFG.save_models = True`:

```text
models/07_logistic_baseline.joblib
models/07_logistic_calibrated_isotonic.joblib
models/07_catboost_baseline.cbm
```

In [ ]:
if CFG.save_models:
    if joblib is None:
        print("joblib não instalado. Modelos sklearn não serão salvos.")
    else:
        joblib.dump(baseline_pipe, MODEL_DIR / "07_logistic_baseline.joblib")
        joblib.dump(calibrated_logistic, MODEL_DIR / "07_logistic_calibrated_isotonic.joblib")
        print("Modelos sklearn salvos em:", MODEL_DIR)

    if "cat_model" in globals():
        try:
            cat_model.save_model(str(MODEL_DIR / "07_catboost_baseline.cbm"))
            print("Modelo CatBoost salvo.")
        except Exception as e:
            print("Não foi possível salvar CatBoost:", e)
else:
    print("Salvamento de modelos desabilitado.")

## 21. MLflow opcional

Use esta seção apenas se o ambiente já tiver MLflow configurado.

Por padrão:

```python
CFG.enable_mlflow = False
```

A ideia é registrar:

- parâmetros do notebook;
- métricas dos modelos;
- artefatos CSV;
- figuras;
- modelos.

In [ ]:
if CFG.enable_mlflow:
    try:
        import mlflow
        import mlflow.sklearn

        mlflow.set_experiment("jurimetria_target_banco_ganhou_baseline")

        with mlflow.start_run(run_name="07_modelagem_baseline_target_banco_ganhou"):
            mlflow.log_params(config_dict)

            for _, row in metrics_summary.iterrows():
                model_name = row["model"]
                for col, value in row.items():
                    if col == "model":
                        continue
                    if isinstance(value, (int, float, np.integer, np.floating)):
                        mlflow.log_metric(f"{model_name}__{col}", float(value))

            for artifact_path in [
                REPORT_DIR / "07_all_models_metrics_summary.csv",
                REPORT_DIR / "07_logistic_financial_gain_table.csv",
                REPORT_DIR / "07_logistic_calibrated_financial_gain_table.csv",
                SCORE_DIR / "07_logistic_ranked_processes_test.csv",
                SCORE_DIR / "07_logistic_calibrated_ranked_processes_test.csv",
            ]:
                if artifact_path.exists():
                    mlflow.log_artifact(str(artifact_path))

            for fig_file in FIG_DIR.glob("07_*.png"):
                mlflow.log_artifact(str(fig_file), artifact_path="figures")

            mlflow.sklearn.log_model(baseline_pipe, artifact_path="model_logistic_baseline")
            mlflow.sklearn.log_model(calibrated_logistic, artifact_path="model_logistic_calibrated")

        print("Run MLflow registrada com sucesso.")

    except ImportError:
        print("MLflow não instalado. Para usar: pip install mlflow")
    except Exception as e:
        print("Erro ao registrar no MLflow:", e)
else:
    print("MLflow desabilitado. Para habilitar, ajuste CFG.enable_mlflow = True.")

## 22. Checklist de validação antes de levar ao jurídico

Antes de apresentar resultados, valide:

### Target

- Acordo genérico deve ser perda, ganho ou removido?
- Acordo pós-sentença é perda?
- Acordo em execução é perda?
- Extinção sem mérito é sempre êxito?
- Prescrição/decadência são sempre êxito?
- Limpeza de base e baixa administrativa foram removidas?

### Dados

- O split temporal representa o uso real do modelo?
- Todas as features existiriam na data de predição?
- Alguma variável de fase/status carrega informação posterior ao desfecho?
- O join Benner + DeepLegal está auditável?
- Há duplicidade de processos?

### Métricas

- ROC AUC é aceitável?
- PR AUC da classe perda é melhor que a prevalência?
- Top 5% e top 10% capturam uma parcela relevante das perdas?
- Probabilidades estão calibradas?
- Performance é estável por ano, produto, assunto e UF?

### Negócio

- O ranking faz sentido para especialistas jurídicos?
- Os principais grupos de risco são acionáveis?
- O modelo prioriza processos com materialidade relevante?
- O jurídico entende que o score é apoio à decisão, não decisão automática?

## 23. Próximos passos recomendados

1. Rodar este notebook com o Cenário A.
2. Verificar se há erros de colunas ausentes ou memória.
3. Avaliar o baseline estatístico.
4. Avaliar o ganho financeiro no top-k.
5. Repetir com Cenário B, deixando claro que é **score atual com cautela temporal**.
6. Rodar CatBoost, se possível.
7. Criar notebook `08_avaliacao_financeira_modelo_juridico.ipynb`.
8. Criar relatório executivo com:
   - objetivo;
   - regras de target;
   - controles de leakage;
   - performance;
   - top-k financeiro;
   - exemplos de processos ranqueados;
   - limitações;
   - próximos passos.